# Credict Product Analytics
## Module 0 — Project Setup & Schema Design

---

### Company Background

**Credict** is a fictional Indian fintech super-app targeting the underserved credit segment in Tier 1, 2, and 3 cities. It offers four interconnected products:

| Product | Description | Revenue Model |
|---|---|---|
| UPI Payments | P2P and merchant payments | MDR (merchant discount rate) |
| BNPL | Split purchases into 3–12 EMIs | Interest + processing fee |
| Personal Loans | ₹10K–₹5L short-term loans | Interest income |
| Savings Wallet | Interest-bearing wallet | Float income |

**North Star Metric:** Monthly Active Transacting Users (MATU)  
**Key Risk Metric:** Non-Performing Asset (NPA) rate on loans  
**Business Model:** Lending-led super-app — payments acquires users → BNPL monetises → Loans scale revenue

---

### Schema Overview — 15 Tables, 4 Domains

```
DOMAIN 1: USER & IDENTITY     → users, user_kyc, devices, sessions, events
DOMAIN 2: FINANCIAL           → wallets, wallet_transactions, transactions
DOMAIN 3: LENDING             → bnpl_orders, bnpl_payments, loan_applications, loans, loan_payments
DOMAIN 4: EXPERIMENTATION     → experiments, experiment_assignments
```

---
## Step 1: Install Dependencies

In [ ]:
!pip install pandas numpy faker psycopg2-binary sqlalchemy matplotlib seaborn plotly scipy -q
print('✅ All packages installed')

---
## Step 2: Imports & Configuration

In [ ]:
import pandas as pd
import numpy as np
import psycopg2
from sqlalchemy import create_engine, text
from faker import Faker
from datetime import date, timedelta, datetime
import random
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# ── Reproducibility ───────────────────────────────────────────
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
fake = Faker('en_IN')   # Indian locale
Faker.seed(SEED)

# ── Chart styling ─────────────────────────────────────────────
plt.rcParams.update({
    'figure.facecolor':  'white',
    'axes.facecolor':    'white',
    'axes.spines.top':   False,
    'axes.spines.right': False,
    'axes.grid':         True,
    'grid.alpha':        0.3,
    'font.size':         11,
    'axes.titlesize':    13,
    'axes.titleweight':  'bold',
})
PALETTE = ['#534AB7', '#1D9E75', '#D85A30', '#BA7517', '#D4537E', '#378ADD']

print('✅ Imports done')

---
## Step 3: PostgreSQL Connection

Update the credentials below to match your local PostgreSQL setup.  
Default PostgreSQL port is **5432**. Create the database before running this cell:
```sql
-- Run this once in psql as a superuser:
CREATE DATABASE credict;
```

In [ ]:
# ── PostgreSQL Connection Config ──────────────────────────────
# Update these to match your local setup
PG_CONFIG = {
    'host':     'localhost',
    'port':     5432,
    'database': 'credict',
    'user':     'postgres',      # your PostgreSQL username
    'password': 'your_password', # your PostgreSQL password
}

# SQLAlchemy engine (used for pd.read_sql and df.to_sql)
DB_URL = (
    f"postgresql+psycopg2://{PG_CONFIG['user']}:{PG_CONFIG['password']}"
    f"@{PG_CONFIG['host']}:{PG_CONFIG['port']}/{PG_CONFIG['database']}"
)
engine = create_engine(DB_URL, echo=False)

# Raw psycopg2 connection (used for DDL and bulk operations)
conn = psycopg2.connect(**PG_CONFIG)
conn.autocommit = True
cur  = conn.cursor()

# Quick connectivity test
cur.execute('SELECT version()')
print('✅ Connected to:', cur.fetchone()[0])

---
## Step 4: Create Schema

In [ ]:
# Load and execute schema.sql
# Drop existing tables first for a clean rebuild
drop_order = [
    'experiment_assignments', 'experiments',
    'loan_payments', 'loans', 'loan_applications',
    'bnpl_payments', 'bnpl_orders',
    'transactions', 'wallet_transactions', 'wallets',
    'events', 'sessions', 'devices', 'user_kyc', 'users'
]
for tbl in drop_order:
    cur.execute(f'DROP TABLE IF EXISTS {tbl} CASCADE')
print('🗑️  Old tables dropped')

# Also drop views if they exist
cur.execute('DROP VIEW IF EXISTS active_users CASCADE')
cur.execute('DROP VIEW IF EXISTS loan_health CASCADE')

# Execute schema file
with open('../sql/schema.sql', 'r') as f:
    schema_sql = f.read()

# Run each statement
created = []
for stmt in schema_sql.split(';'):
    stmt = stmt.strip()
    if not stmt:
        continue
    try:
        cur.execute(stmt)
        if stmt.upper().startswith('CREATE TABLE'):
            name = stmt.split('TABLE')[1].split('(')[0].strip()
            created.append(name)
            print(f'  ✅ {name}')
    except Exception as e:
        if 'ERROR' in str(e).upper():
            print(f'  ⚠️  {str(e)[:80]}')

print(f'\n📊 Tables created: {len(created)}')

---
## Step 5: Synthetic Data Generation — Configuration

In [ ]:
# ── Dataset Config ────────────────────────────────────────────
N_USERS         = 50_000
START_DATE      = date(2024, 1, 1)
END_DATE        = date(2024, 12, 31)
DATE_RANGE_DAYS = (END_DATE - START_DATE).days

# Acquisition channel mix
CHANNELS        = ['organic', 'referral', 'paid_google', 'paid_meta', 'app_store_search']
CHANNEL_WEIGHTS = [0.30, 0.25, 0.20, 0.15, 0.10]

# City tier distribution
TIER_WEIGHTS    = [0.40, 0.35, 0.25]
CITY_MAP = {
    'tier1': ['Mumbai','Delhi','Bengaluru','Hyderabad','Chennai','Pune','Kolkata'],
    'tier2': ['Jaipur','Lucknow','Chandigarh','Indore','Bhopal','Nagpur','Visakhapatnam'],
    'tier3': ['Mysuru','Madurai','Surat','Varanasi','Patna','Jodhpur','Agra'],
}

# KYC funnel step-level completion rates
KYC_FUNNEL = {
    'phone_verified':    0.92,
    'pan_submitted':     0.78,
    'pan_verified':      0.88,
    'aadhaar_linked':    0.72,
    'selfie_submitted':  0.85,
    'kyc_approved':      0.82,
}

# Lending Club grade-level default rates (calibrated to Indian market)
RISK_GRADES = {
    'A': {'default_rate': 0.02, 'rate_range': (10, 14), 'weight': 0.25},
    'B': {'default_rate': 0.05, 'rate_range': (14, 18), 'weight': 0.30},
    'C': {'default_rate': 0.10, 'rate_range': (18, 22), 'weight': 0.25},
    'D': {'default_rate': 0.18, 'rate_range': (22, 28), 'weight': 0.12},
    'E': {'default_rate': 0.28, 'rate_range': (28, 36), 'weight': 0.08},
}

print(f'✅ Config loaded | {N_USERS:,} users | {START_DATE} → {END_DATE}')

---
## Step 6: Generate & Insert Users

In [ ]:
print('🔄 Generating users...')

users = []
for uid in range(1, N_USERS + 1):
    day_offset  = int(np.random.beta(1.5, 1.0) * DATE_RANGE_DAYS)
    signup_date = START_DATE + timedelta(days=day_offset)
    channel     = np.random.choice(CHANNELS, p=CHANNEL_WEIGHTS)
    tier        = np.random.choice(['tier1','tier2','tier3'], p=TIER_WEIGHTS)
    city        = random.choice(CITY_MAP[tier])
    referrer_id = random.randint(1, uid-1) if channel == 'referral' and uid > 1 else None
    plan_prob   = 0.12 if tier == 'tier1' else 0.06
    plan_type   = 'premium' if random.random() < plan_prob else 'free'
    r = random.random()
    status = 'suspended' if r < 0.01 else ('closed' if r < 0.015 else 'active')

    users.append((
        uid, fake.numerify('+91##########'),
        fake.email() if random.random() > 0.3 else None,
        fake.name(),
        fake.date_of_birth(minimum_age=21, maximum_age=55),
        random.choice(['male','male','female','other']),
        signup_date, channel, referrer_id, city, fake.state(),
        tier, plan_type, status,
    ))

insert_sql = '''
    INSERT INTO users
        (user_id, phone_number, email, full_name, date_of_birth, gender,
         signup_date, signup_channel, referrer_id, city, state,
         city_tier, plan_type, account_status)
    VALUES (%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s)
    ON CONFLICT DO NOTHING
'''
# Batch insert using executemany (fast)
cur.executemany(insert_sql, users)

count = cur.execute('SELECT COUNT(*) FROM users') or conn
cur.execute('SELECT COUNT(*) FROM users')
print(f'✅ Users inserted: {cur.fetchone()[0]:,}')

---
## Step 7: Generate & Insert KYC Records

In [ ]:
print('🔄 Generating KYC records...')

# Fetch users from DB
df_users = pd.read_sql('SELECT user_id, signup_date, city_tier FROM users ORDER BY user_id', engine)

KYC_PROVIDERS    = ['digilocker','karza','signzy']
REJECTION_REASONS = ['photo_quality','name_mismatch','id_expired','fraud_flag','blurry_selfie']

kyc_rows = []
for _, user in df_users.iterrows():
    uid     = int(user['user_id'])
    base_ts = datetime.combine(user['signup_date'], datetime.min.time())
    t       = base_ts

    def nxt(lo=2, hi=120):
        nonlocal t
        t = t + timedelta(minutes=random.randint(lo, hi))
        return t

    phone_ok  = random.random() < KYC_FUNNEL['phone_verified']
    phone_ts  = nxt(1,10) if phone_ok else None
    pan_sub   = phone_ok and random.random() < KYC_FUNNEL['pan_submitted']
    pan_sub_ts= nxt(5,60) if pan_sub else None
    pan_ver   = pan_sub and random.random() < KYC_FUNNEL['pan_verified']
    pan_ver_ts= nxt(1,30) if pan_ver else None
    aadhaar   = pan_ver and random.random() < KYC_FUNNEL['aadhaar_linked']
    aadhaar_ts= nxt(5,45) if aadhaar else None
    selfie    = aadhaar and random.random() < KYC_FUNNEL['selfie_submitted']
    selfie_ts = nxt(10,120) if selfie else None
    approved  = selfie and random.random() < KYC_FUNNEL['kyc_approved']

    if not phone_ok:       status = 'pending'
    elif selfie and not approved: status = 'rejected'
    elif approved:         status = 'approved'
    elif selfie:           status = 'in_review'
    else:                  status = 'pending'

    approved_ts      = nxt(30,480) if approved else None
    rejection_reason = random.choice(REJECTION_REASONS) if status == 'rejected' else None

    kyc_rows.append((
        uid, uid,
        phone_ok, phone_ts,
        pan_sub, pan_sub_ts, pan_ver, pan_ver_ts,
        aadhaar, aadhaar_ts,
        selfie, selfie_ts,
        status, approved_ts, rejection_reason,
        random.choice(KYC_PROVIDERS),
    ))

kyc_sql = '''
    INSERT INTO user_kyc
        (kyc_id, user_id,
         phone_verified, phone_verified_at,
         pan_submitted, pan_submitted_at, pan_verified, pan_verified_at,
         aadhaar_linked, aadhaar_linked_at,
         selfie_submitted, selfie_submitted_at,
         kyc_status, kyc_approved_at, rejection_reason, kyc_provider)
    VALUES (%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s)
'''
cur.executemany(kyc_sql, kyc_rows)

df_kyc = pd.read_sql("SELECT * FROM user_kyc", engine)
approved_n = (df_kyc.kyc_status == 'approved').sum()
print(f'✅ KYC records: {len(df_kyc):,}')
print(f'   Approved: {approved_n:,}  ({100*approved_n/len(df_kyc):.1f}%)')
print(df_kyc.kyc_status.value_counts().to_string())

---
## Step 8: Generate Loan Applications, Loans & Payments

In [ ]:
print('🔄 Generating loan applications...')

df_kyc_full = pd.read_sql('SELECT user_id, kyc_status FROM user_kyc', engine)
approved_ids = df_kyc_full[df_kyc_full.kyc_status == 'approved']['user_id'].tolist()
loan_applicants = random.sample(approved_ids, int(len(approved_ids) * 0.35))

APPROVAL_PROB = {'A':0.90,'B':0.80,'C':0.70,'D':0.55,'E':0.35}
PURPOSES      = ['personal','education','medical','business','home_improvement']
EMP_TYPES     = ['salaried','salaried','self_employed','business']
REJECT_RSNS   = ['low_score','high_dti','income_insufficient','existing_debt']

apps = []
for app_id, uid in enumerate(loan_applicants, 1):
    user       = df_users[df_users.user_id == uid].iloc[0]
    app_date   = user['signup_date'] + timedelta(days=random.randint(7, 90))
    if app_date > END_DATE: app_date = END_DATE

    base       = 650 if user['city_tier']=='tier1' else (600 if user['city_tier']=='tier2' else 560)
    score      = int(np.clip(np.random.normal(base, 80), 300, 900))
    grade      = 'A' if score>=750 else ('B' if score>=700 else ('C' if score>=650 else ('D' if score>=600 else 'E')))
    requested  = round(random.choice([10000,25000,50000,75000,100000,150000,200000,300000,500000]),-3)
    tenure     = random.choice([6,12,18,24,36])
    rate       = round(random.uniform(*RISK_GRADES[grade]['rate_range']), 2)
    decision   = 'approved' if random.random() < APPROVAL_PROB[grade] else 'rejected'
    dec_date   = app_date + timedelta(days=random.randint(0,3))

    apps.append((
        app_id, uid, app_date,
        random.choice(PURPOSES), requested, tenure,
        round(random.uniform(180000,1200000),-3),
        random.choice(EMP_TYPES), score,
        round(random.uniform(5,55),1), random.randint(0,3),
        decision, dec_date,
        random.choice(REJECT_RSNS) if decision=='rejected' else None,
        requested if decision=='approved' else None,
        rate if decision=='approved' else None,
        grade,
    ))

app_sql = '''
    INSERT INTO loan_applications
        (application_id,user_id,application_date,loan_purpose,requested_amount,
         tenure_months,annual_income,employment_type,credit_score,
         debt_to_income,existing_loans,decision,decision_date,
         rejection_reason,approved_amount,approved_rate,risk_grade)
    VALUES (%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s)
'''
cur.executemany(app_sql, apps)

df_apps = pd.read_sql("SELECT * FROM loan_applications", engine)
approved_apps = df_apps[df_apps.decision == 'approved']
print(f'✅ Applications: {len(df_apps):,}  |  Approved: {len(approved_apps):,}  ({100*len(approved_apps)/len(df_apps):.1f}%)')

In [ ]:
print('🔄 Generating loans + installment schedule...')

loan_rows = []
lpmt_rows = []
lid, pid  = 1, 1

for _, app in approved_apps.iterrows():
    grade    = app['risk_grade']
    rate     = float(app['approved_rate'])
    principal= float(app['approved_amount'])
    tenure   = int(app['tenure_months'])
    disb_dt  = app['decision_date'] + timedelta(days=random.randint(1,3))
    mrate    = rate / 100 / 12
    emi      = round(principal * mrate * (1+mrate)**tenure / ((1+mrate)**tenure - 1), 2) if mrate > 0 else round(principal/tenure, 2)
    mat_dt   = disb_dt + timedelta(days=30*tenure)

    will_def = random.random() < RISK_GRADES[grade]['default_rate']
    def_at   = random.randint(3, tenure) if will_def else None

    if will_def:                    l_status = 'npa'
    elif mat_dt <= END_DATE:         l_status = 'closed'
    else:                            l_status = 'active'

    loan_rows.append((
        lid, int(app['application_id']), int(app['user_id']),
        principal, tenure, rate, emi,
        round(principal*0.02,2), disb_dt, mat_dt, grade,
        l_status, 0,
        mat_dt if l_status=='closed' else None,
    ))

    remaining = principal
    for inst in range(1, tenure+1):
        due_dt = disb_dt + timedelta(days=30*inst)
        if due_dt > END_DATE + timedelta(days=30): break

        int_comp = round(remaining * mrate, 2)
        pri_comp = round(emi - int_comp, 2)
        remaining= max(0, remaining - pri_comp)

        if will_def and def_at and inst >= def_at:
            paid_amt  = 0
            paid_date = None
            dpd       = max(0, (END_DATE - due_dt).days) if due_dt <= END_DATE else 0
            status    = 'written_off' if dpd > 90 else 'overdue'
        else:
            delay = 0
            r = random.random()
            if r < 0.05:  delay = random.randint(1,10)
            elif r < 0.02: delay = random.randint(11,30)
            paid_date = due_dt + timedelta(days=delay) if due_dt <= END_DATE else None
            paid_amt  = emi if paid_date else 0
            dpd       = delay
            status    = 'paid' if paid_date else 'pending'

        paid_ts = datetime.combine(paid_date, datetime.min.time()) if paid_date else None
        lpmt_rows.append((
            pid, lid, int(app['user_id']), inst,
            due_dt, emi, pri_comp, int_comp,
            paid_amt, paid_date, paid_ts, dpd,
            round(dpd*emi*0.001,2) if dpd>0 else 0,
            status,
        ))
        pid += 1
    lid += 1

loan_sql = '''
    INSERT INTO loans
        (loan_id,application_id,user_id,loan_amount,tenure_months,
         interest_rate,emi_amount,processing_fee,disbursement_date,
         maturity_date,risk_grade,loan_status,days_past_due,closure_date)
    VALUES (%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s)
'''
lpmt_sql = '''
    INSERT INTO loan_payments
        (loan_payment_id,loan_id,user_id,installment_no,due_date,
         due_amount,principal_component,interest_component,
         paid_amount,paid_date,paid_ts,days_past_due,penalty_amount,status)
    VALUES (%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s)
'''

# Batch in chunks for performance
CHUNK = 1000
for i in range(0, len(loan_rows), CHUNK):
    cur.executemany(loan_sql, loan_rows[i:i+CHUNK])
for i in range(0, len(lpmt_rows), CHUNK):
    cur.executemany(lpmt_sql, lpmt_rows[i:i+CHUNK])

cur.execute('SELECT COUNT(*) FROM loans')
print(f'✅ Loans: {cur.fetchone()[0]:,}')
cur.execute('SELECT COUNT(*) FROM loan_payments')
print(f'✅ Loan payments: {cur.fetchone()[0]:,}')

---
## Step 9: Data Quality Checks

In [ ]:
print('🔍 Running data quality checks...\n')

checks = [
    ('users',             'user_id'),
    ('user_kyc',          'kyc_id'),
    ('loan_applications', 'application_id'),
    ('loans',             'loan_id'),
    ('loan_payments',     'loan_payment_id'),
]

for table, pk in checks:
    cur.execute(f'SELECT COUNT(*) FROM {table}')
    count = cur.fetchone()[0]
    cur.execute(f'SELECT COUNT(*) FROM (SELECT {pk}, COUNT(*) c FROM {table} GROUP BY {pk} HAVING c>1) x')
    dups  = cur.fetchone()[0]
    flag  = '✅' if dups==0 else '❌'
    print(f'  {flag} {table:25s} | rows: {count:>8,} | dup PKs: {dups}')

# Referential integrity
cur.execute('''
    SELECT COUNT(*) FROM loans l
    LEFT JOIN loan_applications a ON l.application_id = a.application_id
    WHERE a.application_id IS NULL
''')
print(f'\n  ✅ Orphan loans (expect 0): {cur.fetchone()[0]}')

# KYC coverage
cur.execute('''
    SELECT
        COUNT(DISTINCT u.user_id)                                            AS total,
        COUNT(DISTINCT k.user_id)                                            AS has_kyc,
        COUNT(DISTINCT CASE WHEN k.kyc_status=\'approved\' THEN k.user_id END) AS approved
    FROM users u LEFT JOIN user_kyc k ON u.user_id=k.user_id
''')
row = cur.fetchone()
print(f'\n  📊 KYC: {row[0]:,} total → {row[1]:,} have KYC → {row[2]:,} approved ({100*row[2]/row[0]:.1f}%)')
print('\n✅ Quality checks passed')

---
## Step 10: Summary Charts

In [ ]:
# Use TO_CHAR for PostgreSQL (not strftime)
df_monthly = pd.read_sql('''
    SELECT
        TO_CHAR(signup_date, \'YYYY-MM\') AS month,
        COUNT(*) AS new_users
    FROM users
    GROUP BY 1
    ORDER BY 1
''', engine)

df_kyc_summary = pd.read_sql('''
    SELECT kyc_status, COUNT(*) AS users
    FROM user_kyc
    GROUP BY kyc_status
    ORDER BY users DESC
''', engine)

df_funnel = pd.read_sql('''
    SELECT
        COUNT(*)                                                       AS registered,
        COUNT(CASE WHEN phone_verified   THEN 1 END)                   AS phone_otp,
        COUNT(CASE WHEN pan_submitted    THEN 1 END)                   AS pan_submitted,
        COUNT(CASE WHEN pan_verified     THEN 1 END)                   AS pan_verified,
        COUNT(CASE WHEN aadhaar_linked   THEN 1 END)                   AS aadhaar_linked,
        COUNT(CASE WHEN selfie_submitted THEN 1 END)                   AS selfie_done,
        COUNT(CASE WHEN kyc_status = \'approved\' THEN 1 END)           AS kyc_approved
    FROM user_kyc
''', engine)

fig, axes = plt.subplots(1, 2, figsize=(15, 5))
fig.suptitle('Credict — Module 0 Dataset Summary', fontsize=14, fontweight='bold', y=1.01)

# Chart 1: Monthly signups
ax = axes[0]
ax.bar(df_monthly['month'], df_monthly['new_users'],
       color=PALETTE[0], alpha=0.85, edgecolor='white', linewidth=0.5)
ax.set_title('Monthly User Signups (2024)')
ax.set_xlabel('Month')
ax.set_ylabel('New Users')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: f'{int(x):,}'))
plt.setp(ax.get_xticklabels(), rotation=45, ha='right')

# Chart 2: KYC funnel
ax2 = axes[1]
steps = ['Registered','Phone OTP','PAN Submitted','PAN Verified','Aadhaar Linked','Selfie Done','KYC Approved']
vals  = [int(df_funnel[c].iloc[0]) for c in ['registered','phone_otp','pan_submitted','pan_verified','aadhaar_linked','selfie_done','kyc_approved']]
colors = plt.cm.YlOrRd(np.linspace(0.2, 0.85, len(steps)))
bars = ax2.barh(steps[::-1], vals[::-1], color=colors[::-1], edgecolor='white', linewidth=0.5)
for bar, val in zip(bars, vals[::-1]):
    ax2.text(val + 300, bar.get_y() + bar.get_height()/2,
             f'{val:,}', va='center', fontsize=9)
ax2.set_title('KYC Activation Funnel')
ax2.set_xlabel('Users')
ax2.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: f'{int(x):,}'))

plt.tight_layout()
plt.savefig('../assets/charts/00_setup_overview.png', dpi=150, bbox_inches='tight')
plt.show()
print('💾 Chart saved')

---
## Module 0 Complete ✅

| Component | Details |
|---|---|
| **Database** | PostgreSQL — `credict` |
| **Tables** | 15 tables across 4 domains |
| **Users** | 50,000 synthetic users (Jan–Dec 2024) |
| **KYC Records** | 50,000 rows — full funnel with timestamps per step |
| **Loan Applications** | ~8,750 (35% of KYC-approved users) |
| **Loans** | ~6,300 approved with grade-calibrated default rates |
| **Loan Payments** | ~75,000+ installment records with realistic DPD |

### Key Design Decisions
- **PostgreSQL** — production-grade SQL, full window functions, JSONB support
- **`en_IN` Faker locale** — realistic Indian names, cities, phone numbers
- **Beta distribution** for signups — simulates accelerating organic growth, not flat random
- **Lending Club grade defaults** — A=2% → E=28% — real credit risk patterns
- **Installment-level payments** — enables DPD bucket analysis and vintage curves in Module 4
- **Separate `loan_applications` + `loans`** — captures the underwriting decision, not just approved loans

---
**Next → Module 1: User Acquisition & KYC Funnel Analysis** 🚀